# 0️⃣1️⃣ - Authentication methods
![Persona](https://img.shields.io/badge/%F0%9F%91%A4%20Persona-%F0%9F%A4%96%20Network%20Automation%20Developer-lightgrey) ![Difficulty](https://img.shields.io/badge/%F0%9F%9A%A6%20Difficulty-Beginner-green) ![RADKit version](https://img.shields.io/badge/RADKit-1.9.6-blue?logo=cisco&logoColor=white) ![Python version](https://img.shields.io/badge/Python-3.12%2B-yellow?logo=python&logoColor=white)

> In this playbook we will explore the different ways to authenticate with your RADKit server.

### 🛠️ Before You Begin

If you haven't set up your environment yet, follow the instructions in [SETUP.md](../SETUP.md) before running any cells.

---

### 📋 What you'll learn

| # | Topic |
|---|-------|
| 1 | How to authenticate interactively using **SSO login** (browser-based) |
| 2 | How to enroll your client for certificate-based authentication |
| 3 | How to authenticate non-interactively using **certificate-based login** |
| 4 | How to use environment variables and custom certificate paths for automation |

---

### 🤖 The `Client` Object

In the RADKit SDK, a **Client** is your application-side handle for authentication and session management. In this playbook, we will explore the different authentication methods available to create a `Client`.

> In the following examples, you will see that the Client is always used with a **context manager** — `with Client.create() as client: ...`. This is the recommended pattern to ensure the client session is properly closed after use.

---

## ☁️ Method 1: `sso_login` — Interactive, Web-Based Authentication

**Best for:** Interactive use, development environments, and any scenario where a browser window can be opened.

**How it works:**
1. `sso_login` opens your web browser with the Cisco SSO login form.
2. The script pauses and waits until you complete the login.
3. Once authenticated, execution resumes automatically.

**What you need:**
- Your **CCO user ID** (e.g., `jsmith@cisco.com`)

![SSO Login form](../images/03-sso-auth.png)

In [2]:
from radkit_client import Client
from radkit_client.sync import ClientStatus # Nice enum to get the status of the client

user_id = input("👤 Enter your CCO user id: ")

with Client.create() as client:
    client.sso_login(user_id) # Web form SSO login with remote user's email address (blocking call)
    print("✅ Authentication successful!") if client.status == ClientStatus.CONNECTED else print("🔥 Connection failed ...")


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.
✅ Authentication successful!


---

## 🔑 Method 2: `certificate_login` — Non-Interactive, Certificate-Based Authentication

**Best for:** Automated scripts, CI/CD pipelines, and any scenario where browser interaction is not possible.

**How it works:** Instead of opening a browser, `certificate_login` uses security certificates stored on your local machine to authenticate silently.

> **Prerequisite:** You must first enroll your client to generate the required certificates. See section 2.1 below.

---

### 2.1 Enrolling the client

Before using certificate-based authentication, you need to **enroll your client** once to generate the necessary certificates. This is done by authenticating with SSO and calling `enroll_client()`.

> This code cannot be run in this notebook because interactive password input is not supported in Jupyter. Run [this standalone script](001-enroll-client.py) instead: `python 001-enroll-client.py`

🔑 Once enrolled, your certificates will be stored at: `~/.radkit/identities/`

### 2.2 Authentication with password

Once enrolled, you can authenticate by providing the private key password you set during enrollment:

**What you need:**
- Your **CCO user ID**
- The **private key password** you set during enrollment

In [4]:
from radkit_client import Client
from radkit_client.sync import ClientStatus
import getpass

user_id = input("👤 Enter your CCO user id: ")
private_key_password = getpass.getpass("🔑 Enter the password for your private key: ")

with Client.create() as client:
    client.certificate_login(identity = user_id, private_key_password = private_key_password) # Certificate-based login
    print("✅ Authentication successful!") if client.status == ClientStatus.CONNECTED else print("🔥 Connection failed ...")

✅ Authentication successful!


### 2.3 Authentication with an environment variable

For fully automated workflows where no human input is possible, you can store the password in the environment variable `RADKIT_CLIENT_PRIVATE_KEY_PASSWORD_BASE64`. The RADKit SDK will read it automatically on login, no need to pass the password in code.

> The password must be **base64-encoded** before setting the variable.

**What you need:**
- Your **CCO user ID**
- Your **private key password** (encoded at runtime — not stored in plain text)

The cell below shows how to encode the password and set the environment variable programmatically at runtime:

In [5]:
from radkit_client import Client
from radkit_client.sync import ClientStatus
import getpass
import base64
import os

user_id = input("👤 Enter your CCO user id: ")

private_key_password = getpass.getpass("🔑 Enter the password for your private key: ")
private_key_password_b64 = base64.b64encode(private_key_password.encode()).decode() # Encoding to base64
os.environ["RADKIT_CLIENT_PRIVATE_KEY_PASSWORD_BASE64"] = private_key_password_b64  # Setting of special env var

with Client.create() as client:
    client.certificate_login(user_id) # Certificate-based login with password as env variable
    print("✅ Authentication successful!") if client.status == ClientStatus.CONNECTED else print("🔥 Connection failed ...")

✅ Authentication successful!


### 2.4 Authentication with explicit certificate paths

**Best for:** Containerized environments or any setup where certificates are mounted from a custom location rather than `~/.radkit/identities/`.

**How it works:** Pass the file paths directly to `certificate_login` instead of relying on the default discovery logic.

> The certificate files are generated during client enrollment (section 2.1) and located at:
> `~/.radkit/identities/<your-domain>.radkit-cloud.cisco.com/<your_remote_user>/`

**What you need:**
- Your **CCO user ID** and **private key password**
- Paths to your **certificate.pem**, **chain.pem**, and **private_key_encrypted.pem** files

In [7]:
from radkit_client import Client
from radkit_client.sync import ClientStatus
import getpass

user_id = input("👤 Enter your CCO user id: ")
private_key_password = getpass.getpass("🔑 Enter the password for your private key: ")
cert_path = input("📁 Enter the path to your certificate.pem file: ")
ca_path = input("📁 Enter the path to your chain.pem file: ")
key_path = input("📁 Enter the path to your private_key_encrypted.pem file: ")

with Client.create() as client:
    client.certificate_login(
        identity = user_id,
        private_key_password = private_key_password,
        cert_path = cert_path,
        ca_path = ca_path,
        key_path = key_path
        ) # Certificate-based login using the provided paths
    print("✅ Authentication successful!") if client.status == ClientStatus.CONNECTED else print("🔥 Connection failed ...")

✅ Authentication successful!


---

## 🔀 Which Method Should I Use?

| | ☁️ SSO Login | 🔑 Certificate (password) | 🔑 Certificate (env var) | 🔑 Certificate (custom paths) |
|---|---|---|---|---|
| **Interaction required** | Yes (browser) | Yes (password prompt) | No | Yes (path input) |
| **Enrollment required** | No | Yes | Yes | Yes |
| **Use case** | Getting started, ad-hoc | Semi-automated scripts | CI/CD pipelines | Containers, custom deployments |
| **Password in code** | No | No | No (env var) | No |